# Считаю retrieval метрики по DINOv2 

In [17]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
import os
from dotenv import load_dotenv
from sneakers_hse.data.utils.s3_tools import S3Client
from sklearn.model_selection import train_test_split

load_dotenv()

True

In [19]:
# PROJECT_ROOT_PATH = Path('/Users/alexansemyachkin/Desktop/studies/hse/sneakers-hse')
PROJECT_ROOT_PATH = Path(os.getenv('PROJECT_ROOT_PATH'))
# Скачиваю эмбеддинги из s3
s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
              aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))

In [50]:
s3._download_one(
    bucket_name='sneakers-hse-images-test',
    s3_key='dinov2/embeddings.parquet.gzip', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_path=str(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')
)

'/Users/a.r.makarenko/Documents/hse/sneakers-hse/data/04_dinov2_embeddings.parquet.gzip'

In [20]:
s3.download_folder_from_s3_parallel(
    bucket_name='sneakers-hse-images-test',
    s3_prefix='dinov2/chroma_db', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_folder=str(PROJECT_ROOT_PATH / 'chroma_db'),
    max_workers=10
)

Total files: 27
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/1a6a579a-10d8-4d7c-a0ac-cbe18ec0ee40/link_lists.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/link_lists.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/link_lists.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/header.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/length.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/1a6a579a-10d8-4d7c-a0ac-cbe18ec0ee40/length.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/76efa518-b14e-4991-b2c5-3211233aa8ef/header.bin
Downloaded: /Users/a.r.makarenko/Documents/hse/sneakers-hse/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/length.bin
Downloaded: /Users/a.r.makar

In [3]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()

class,count
str,u32
"""adidas Кроссовки GALAXY 7 M""",288
"""adidas Кроссовки DURAMO SL2 M""",496
"""PUMA Кроссовки RS-X Efekt PRM""",260
"""Vans Кеды Hylane""",275
"""Maison David Кроссовки""",286
…,…
"""Nike Кеды FIELD GENERAL""",274
"""Under Armour Кроссовки UA CHAR…",287
"""Vans Кеды Upland""",275


In [4]:
train_df_pre = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
test_df = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/test_images.csv')

train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
2288,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0
10855,"""PUMA Кроссовки Puma Morphic/or…","""PUMA Кроссовки Puma Morphic""",0
7477,"""Kari Кроссовки/clothing_0_190.…","""Kari Кроссовки""",0
4230,"""Reebok Кроссовки CLASSIC LEATH…","""Reebok Кроссовки CLASSIC LEATH…",0
2188,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0


(8772, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
7639,"""Vans Кеды Knu Skool/clothing_0…","""Vans Кеды Knu Skool""",0
2591,"""Reebok Кроссовки CLASSIC NYLON…","""Reebok Кроссовки CLASSIC NYLON""",0
5027,"""Nike Кроссовки AIR MAX 90/clot…","""Nike Кроссовки AIR MAX 90""",0
2762,"""Under Armour Кроссовки UA CHAR…","""Under Armour Кроссовки UA CHAR…",0
240,"""Vans Кеды Upland/clothing_0_62…","""Vans Кеды Upland""",0


(2193, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
14,"""Vans Кеды Upland/clothing_0_16…","""Vans Кеды Upland""",0
32,"""Vans Кеды Upland/clothing_0_21…","""Vans Кеды Upland""",0
44,"""Vans Кеды Upland/orig_216_real…","""Vans Кеды Upland""",0
80,"""Vans Кеды Upland/shoe_3_100_re…","""Vans Кеды Upland""",0
87,"""Vans Кеды Upland/clothing_0_27…","""Vans Кеды Upland""",0


(300, 4)

In [5]:
sql = pl.SQLContext()
sql.register_globals()
df = sql.execute(
    '''
select
    df.*
    , case when train_df.path is not null then 'train'
            when val_df.path is not null then 'val'
            when test_df.path is not null then 'test'
            end as sample_part
from df
left join train_df using(path)
left join val_df using(path)
left join test_df using(path)
'''
).collect()
df

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train"""
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train"""
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train"""
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train"""
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train"""
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]","""train"""


In [6]:
client = chromadb.Client()
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})

def add_in_batches(collection, ids, embeddings, metadatas, batch_size=5000):
    for i in range(0, len(ids), batch_size):
        collection.add(
            ids=ids[i:i+batch_size],
            embeddings=embeddings[i:i+batch_size],
            metadatas=metadatas[i:i+batch_size]
        )

add_in_batches(
    collection,
    ids=paths,
    embeddings=embeddings,
    metadatas=[{"class": c} for c in labels]
)

In [7]:
from tqdm.notebook import tqdm

def get_neighbors(collection, embeddings, k=10, batch_size=10):
    all_results = []
    for i in tqdm(range(0, len(embeddings), batch_size), desc="Querying"):
        batch = embeddings[i:i+batch_size]
        results = collection.query(
            query_embeddings=batch,
            n_results=k + 1  # +1 чтобы убрать self
        )['metadatas']
        all_results.extend(results)
    return np.array([[neighbor['class'] for neighbor in query]
                     for query in all_results])[:, 1:]  # Убираю self-match

def hit_rate_at_k(neighbors, labels, k):
    hits = 0
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        if query_label in retrieved:
            hits += 1
    return hits / len(labels)

def precision_at_k(neighbors, labels, k):
    precisions = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        relevant = (retrieved == query_label).sum()
        precisions.append(relevant / k)
    return np.mean(precisions)

def average_precision(retrieved, query_label, k):
    score = 0.0
    hits = 0
    for i in range(k):
        if retrieved[i] == query_label:
            hits += 1
            score += hits / (i + 1)
    return score / hits if hits > 0 else 0


def map_at_k(neighbors, labels, k):
    scores = []
    for i in range(len(labels)):
        query_label = labels[i]
        retrieved = neighbors[i][:k]
        scores.append(average_precision(retrieved, query_label, k))
    return np.mean(scores)

neighbors = get_neighbors(collection, embeddings, k=20)
len(neighbors[0])
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    print("HitRate:", hit_rate_at_k(neighbors, labels, k))
    print("Precision:", precision_at_k(neighbors, labels, k))
    print("MAP:", map_at_k(neighbors, labels, k))

Querying:   0%|          | 0/1127 [00:00<?, ?it/s]


Metrics @ 1
HitRate: 0.7298712827341323
Precision: 0.7298712827341323
MAP: 0.7298712827341323

Metrics @ 5
HitRate: 0.8919662671992898
Precision: 0.6077940523746116
MAP: 0.7621754944025251

Metrics @ 10
HitRate: 0.9347536617842876
Precision: 0.5364403018197959
MAP: 0.7185387582939999


In [8]:
df = df.with_columns(neighbors=neighbors)
df = df.with_columns(
    pl.struct(["neighbors", "class"]).map_elements(
        lambda row: [
            1 if n == row["class"] else 0
            for n in row["neighbors"][:k]
        ]
    ).alias("hit_flg")
)
df

path,class,embedding,sample_part,neighbors,hit_flg
str,str,list[f64],str,"array[str, 20]",list[i64]
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train""","[""Vans Кеды Hylane"", ""PUMA Кроссовки RS-X Efekt PRM"", … ""PUMA Кроссовки Darter Pro""]","[0, 0, … 0]"
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train""","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""Vans Кеды Upland""]","[1, 1, … 0]"
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train""","[""Nike Кеды NIKE AIR FORCE 1"", ""PUMA Кеды CA Pro Classic II"", … ""Nike Кеды NIKE AIR FORCE 1""]","[0, 0, … 1]"
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train""","[""Vans Кеды Upland"", ""Vans Кеды Upland"", … ""PUMA Кеды Palermo""]","[1, 1, … 0]"
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train""","[""Vans Кеды Hylane"", ""Vans Кеды Upland"", … ""Vans Кеды Hylane""]","[0, 1, … 1]"
…,…,…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train""","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Charged Rogue 5""]","[1, 1, … 0]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train""","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]","""train""","[""Under Armour Кроссовки UA Phantom 4"", ""Under Armour Кроссовки UA Phantom 4"", … ""Under Armour Кроссовки UA Phantom 4""]","[1, 1, … 1]"


In [13]:
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.729871,0.729871


class,hit_at_k,precision_at_k
str,f64,f64
"""PUMA Кроссовки Darter Pro""",0.910035,0.910035
"""Nike Кроссовки AIR MAX 90""",0.902439,0.902439
"""Nike Кроссовки Nike Zoom Vomer…",0.893617,0.893617
"""Nike Кеды Dunk Low Retro""",0.885017,0.885017
"""Nike Кроссовки AIR PEGASUS 200…",0.881295,0.881295
…,…,…
"""Tendance Кроссовки""",0.505535,0.505535
"""adidas Кроссовки DURAMO SL2 M""",0.493952,0.493952
"""X-Plode Кроссовки""",0.446097,0.446097



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.891966,0.607794


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кроссовки Nike Zoom Vomer…",0.971631,0.839007
"""Nike Кеды Dunk Low Retro""",0.965157,0.777003
"""Nike Кроссовки AIR MAX 90""",0.954704,0.83554
"""adidas Кроссовки RUNFALCON 5""",0.954386,0.619649
"""PUMA Кроссовки Puma Morphic""",0.954198,0.803817
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.783394,0.518412
"""Tendance Кроссовки""",0.723247,0.338745
"""X-Plode Кроссовки""",0.713755,0.326394



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.934754,0.53644


class,hit_at_k,precision_at_k
str,f64,f64
"""adidas Кроссовки RUNFALCON 5""",0.985965,0.545614
"""Nike Кроссовки Nike Zoom Vomer…",0.98227,0.786879
"""Reebok Кроссовки RBK PREMIER T…",0.973585,0.546038
"""Nike Кеды Dunk Low Retro""",0.972125,0.699303
"""Under Armour Кроссовки UA Char…",0.971831,0.502113
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.859206,0.470397
"""X-Plode Кроссовки""",0.821561,0.280669
"""Tendance Кроссовки""",0.808118,0.278967


In [15]:
sample_part = 'train'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.742134,0.742134


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кроссовки AIR PEGASUS 200…",0.912844,0.912844
"""PUMA Кроссовки Darter Pro""",0.910314,0.910314
"""Nike Кроссовки Nike Zoom Vomer…",0.902655,0.902655
"""Nike Кроссовки AIR MAX 90""",0.899563,0.899563
"""Nike Кеды Dunk Low Retro""",0.897778,0.897778
…,…,…
"""adidas Кроссовки DURAMO SL2 M""",0.505376,0.505376
"""Tendance Кроссовки""",0.504673,0.504673
"""X-Plode Кроссовки""",0.471963,0.471963



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.896603,0.618217


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кеды Dunk Low Retro""",0.977778,0.798222
"""Nike Кроссовки Nike Zoom Vomer…",0.973451,0.852212
"""PUMA Кроссовки Puma Morphic""",0.966019,0.818447
"""PUMA Кроссовки Darter Pro""",0.959641,0.809865
"""PUMA Кроссовки RS-X Efekt PRM""",0.959596,0.771717
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.785388,0.518721
"""X-Plode Кроссовки""",0.724299,0.340187
"""Tendance Кроссовки""",0.719626,0.336449



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.939352,0.545577


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кроссовки Nike Zoom Vomer…",0.986726,0.79646
"""adidas Кроссовки RUNFALCON 5""",0.982143,0.552232
"""PUMA Кеды Palermo""",0.981308,0.690654
"""Reebok Кроссовки RBK PREMIER T…",0.980296,0.569951
"""PUMA Кроссовки RS-X Efekt PRM""",0.979798,0.655051
…,…,…
"""Under Armour Кроссовки UA CHAR…",0.858447,0.475342
"""X-Plode Кроссовки""",0.827103,0.293458
"""Tendance Кроссовки""",0.808411,0.273832


In [12]:
sample_part = 'val'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.720018,0.720018


class,hit_at_k,precision_at_k
str,f64,f64
"""PUMA Кроссовки Darter Pro""",0.964286,0.964286
"""Nike Кроссовки AIR MAX 90""",0.929825,0.929825
"""Nike Кеды Dunk Low Retro""",0.910714,0.910714
"""Nike Кеды FIELD GENERAL""",0.888889,0.888889
"""Vans Кеды Sport Low""",0.888889,0.888889
…,…,…
"""adidas Кроссовки DURAMO SL2 M""",0.473118,0.473118
"""Kari Кеды""",0.462963,0.462963
"""adidas Кроссовки DROPSET 3 TRA…",0.428571,0.428571



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.901961,0.610944


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кроссовки AIR MAX 90""",0.982456,0.863158
"""Nike Кеды Dunk Low Retro""",0.982143,0.75
"""Maison David Кроссовки""",0.982143,0.692857
"""adidas Кроссовки RUNFALCON 5""",0.982143,0.639286
"""Vans Кеды Hylane""",0.980392,0.611765
…,…,…
"""Nike Кеды NIKE AIR FORCE 1""",0.793103,0.506897
"""Tendance Кроссовки""",0.735849,0.358491
"""X-Plode Кроссовки""",0.685185,0.277778



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.937984,0.54104


class,hit_at_k,precision_at_k
str,f64,f64
"""Nike Кеды Dunk Low Retro""",1.0,0.667857
"""Vans Кеды Hylane""",1.0,0.515686
"""adidas Кроссовки RUNFALCON 5""",1.0,0.546429
"""Nike Кроссовки AIR MAX 90""",0.982456,0.812281
"""Maison David Кроссовки""",0.982143,0.633929
…,…,…
"""Nike Кеды NIKE AIR FORCE 1""",0.87931,0.456897
"""X-Plode Кроссовки""",0.814815,0.235185
"""Tendance Кроссовки""",0.792453,0.311321


In [14]:
sample_part = 'test'
for k in [1, 5, 10]:
    print(f"\nMetrics @ {k}")
    df_k = df.filter(pl.col('sample_part') == sample_part).with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
        .with_columns(
            pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
            hit_at_k = pl.col("hits_k").list.max(),
            precision_at_k = pl.col("hits_k").list.mean()
        )
    display(df_k.select(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ))
    display(df_k.group_by('class').agg(
        pl.col('hit_at_k').mean(),
        pl.col('precision_at_k').mean()
    ).sort(pl.col('hit_at_k'), descending=True))


Metrics @ 1


hit_at_k,precision_at_k
f64,f64
0.443333,0.443333


class,hit_at_k,precision_at_k
str,f64,f64
"""adidas Кроссовки GALAXY 7 M""",1.0,1.0
"""Under Armour Кроссовки UA CHAR…",0.8,0.8
"""Reebok Кроссовки CLASSIC NYLON""",0.727273,0.727273
"""PUMA Кроссовки RS-X Efekt PRM""",0.666667,0.666667
"""adidas Кеды VL COURT 3""",0.666667,0.666667
…,…,…
"""Kari Кеды""",0.0,0.0
"""Kari Кроссовки""",0.0,0.0
"""Under Armour Кроссовки UA Char…",0.0,0.0



Metrics @ 5


hit_at_k,precision_at_k
f64,f64
0.683333,0.28


class,hit_at_k,precision_at_k
str,f64,f64
"""Under Armour Кроссовки UA CHAR…",1.0,0.44
"""X-Plode Кеды""",1.0,0.2
"""adidas Кроссовки GALAXY 7 M""",1.0,0.811765
"""Tendance Кеды""",1.0,0.4
"""PUMA Кеды Palermo""",1.0,0.3
…,…,…
"""X-Plode Кроссовки""",0.0,0.0
"""Under Armour Кроссовки UA Char…",0.0,0.0
"""Kari Кеды""",0.0,0.0



Metrics @ 10


hit_at_k,precision_at_k
f64,f64
0.776667,0.235667


class,hit_at_k,precision_at_k
str,f64,f64
"""Tendance Кеды""",1.0,0.5
"""X-Plode Кеды""",1.0,0.2
"""PUMA Кеды Palermo""",1.0,0.15
"""Under Armour Кроссовки UA CHAR…",1.0,0.34
"""adidas Кроссовки GALAXY 7 M""",1.0,0.741176
…,…,…
"""Nike Кроссовки AIR MAX 90""",0.0,0.0
"""Kari Кеды""",0.0,0.0
"""Under Armour Кроссовки UA Phan…",0.0,0.0
